# VN2 Inventory Forecasting with KumoRFM

## Overview

This notebook leverages **KumoRFM (Kumo Relational Foundation Model)** to generate demand forecasts for the VN2 inventory optimization challenge. KumoRFM is a Foundation Model for machine learning on enterprise data that requires no model training—just data and a few lines of code.

### Why KumoRFM for Inventory Forecasting?

1. **Multi-table reasoning**: KumoRFM naturally handles the relational structure of our data (Stores, Products, Sales, Availability, Product Hierarchy)
2. **Temporal awareness**: Automatically models how demand evolves over time using timestamps
3. **Zero-shot predictions**: No training required—the pre-trained foundation model generalizes from graph structure
4. **Censorship-aware**: We can mask out-of-stock weeks in our graph structure
5. **Predictive Query Language (PQL)**: Express forecasting tasks naturally (e.g., "predict demand in next 3 weeks")

## Requirements

- KumoAI SDK (`pip install kumoai --pre --upgrade`)
- KumoRFM API key (free tier available)
- VN2 Week 0 data files

## Setup: Install and Initialize KumoRFM


In [1]:
# Install KumoAI SDK
%pip install kumoai --pre --upgrade -q



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import kumoai.experimental.rfm as rfm

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Paths
DATA_DIR = PROJECT_ROOT / "data"
SUB_DIR = PROJECT_ROOT / "submissions"
SUB_DIR.mkdir(exist_ok=True)

print(f"✅ KumoRFM SDK loaded")
print(f"📁 Data directory: {DATA_DIR}")


✅ KumoRFM SDK loaded
📁 Data directory: /Users/senoni/noni/vn2inventory/data


In [3]:
os.environ["KUMO_API_KEY"] = "REDACTED_KUMO_API_KEY"
# Authenticate with KumoRFM (free API key)
# This will open a widget to generate/retrieve your API key
if not os.environ.get("KUMO_API_KEY"):
    rfm.authenticate()

# Initialize KumoRFM client
rfm.init()
print("✅ KumoRFM client initialized")


[2025-10-09 14:52:06 - kumoai:203 - INFO] Successfully initialized the Kumo SDK against deployment https://kumorfm.ai/api, with log level INFO.


✅ KumoRFM client initialized


## Load VN2 Data and Apply Censorship Masking

We'll load the data and mask out-of-stock (OOS) weeks to ensure KumoRFM doesn't learn from censored zeros.


In [4]:
# Load VN2 Week 0 files
INDEX = ["Store", "Product"]

sales_wide = pd.read_csv(DATA_DIR / "Week 0 - 2024-04-08 - Sales.csv")
avail_wide = pd.read_csv(DATA_DIR / "Week 0 - In Stock.csv")
master = pd.read_csv(DATA_DIR / "Week 0 - Master.csv")
initial_state = pd.read_csv(DATA_DIR / "Week 0 - 2024-04-08 - Initial State.csv")
template = pd.read_csv(DATA_DIR / "Week 0 - Submission Template.csv")

print(f"📊 Loaded {len(sales_wide)} SKUs, {len(sales_wide.columns)-2} weeks of sales history")


📊 Loaded 599 SKUs, 157 weeks of sales history


In [5]:
# Availability-aware masking: OOS weeks -> NaN
sw = sales_wide.set_index(INDEX).copy()
aw = avail_wide.set_index(INDEX).copy()

# Align columns to datetime
sw.columns = pd.to_datetime(sw.columns)
aw.columns = pd.to_datetime(aw.columns)

# Mask: sales_c has NaN when out of stock, true zeros preserved
avail_mask = aw.astype(bool)
sales_c = sw.where(avail_mask)

# QA: ensure no censored zeros leak
assert sales_c.where(~avail_mask).isna().all().all(), "Censored weeks must be NaN"

print(f"✅ Applied availability-aware masking")
print(f"   Available observations: {sales_c.notna().sum().sum():,}")
print(f"   Censored (OOS): {(~avail_mask).sum().sum():,}")


✅ Applied availability-aware masking
   Available observations: 83,526
   Censored (OOS): 10,517


## Create Entity Tables for KumoRFM Graph

KumoRFM works with relational graphs. We'll create:
- **Stores**: unique store entities with optional hierarchy (Region, Cluster)
- **Products**: product catalog with Division/Department/ProductGroup hierarchy
- **Sales**: temporal events (store_id, product_id, week, qty) - censorship-aware


In [6]:
# 1. STORES table
stores_df = pd.DataFrame({
    'store_id': sales_wide['Store'].unique()
})

# Optional: add store hierarchy if available in Master
if 'Region' in master.columns or 'Cluster' in master.columns:
    store_attrs = master[['Store'] + [c for c in ['Region', 'Cluster'] if c in master.columns]].drop_duplicates()
    stores_df = stores_df.merge(
        store_attrs.rename(columns={'Store': 'store_id'}),
        on='store_id',
        how='left'
    )

stores_df = stores_df.drop_duplicates('store_id').reset_index(drop=True)
print(f"✅ STORES: {len(stores_df)} stores")
display(stores_df.head())


✅ STORES: 67 stores


,store_id
0,0
1,1
2,2
3,3
4,4


In [7]:
# 2. PRODUCTS table with hierarchy
products_df = master[['Product', 'Division', 'Department', 'ProductGroup']].copy()
products_df = products_df.rename(columns={'Product': 'product_id'})
products_df = products_df.drop_duplicates('product_id').reset_index(drop=True)

print(f"✅ PRODUCTS: {len(products_df)} products")
print(f"   Divisions: {products_df['Division'].nunique()}")
print(f"   Departments: {products_df['Department'].nunique()}")
print(f"   Product Groups: {products_df['ProductGroup'].nunique()}")
display(products_df.head())


✅ PRODUCTS: 297 products
   Divisions: 47
   Departments: 26
   Product Groups: 111


,product_id,Division,Department,ProductGroup
0,126,3012,30,301202
1,182,4404,44,440403
2,124,2402,24,240201
3,49,402,4,40215
4,103,3012,30,301202


In [8]:
# 3. SALES table (long format, censorship-aware)
# Melt sales_c (availability-masked) to long format
sales_long = (
    sales_c.reset_index()
    .melt(id_vars=INDEX, var_name='week', value_name='qty')
    .rename(columns={'Store': 'store_id', 'Product': 'product_id'})
)

# Drop NaN (censored weeks) - KumoRFM will not see these events
sales_long = sales_long.dropna(subset=['qty'])

# Ensure correct types
sales_long['qty'] = sales_long['qty'].astype(float)
sales_long['week'] = pd.to_datetime(sales_long['week'])

# Create unique sales_id for primary key
sales_long['sales_id'] = range(len(sales_long))

# Reorder
sales_long = sales_long[['sales_id', 'store_id', 'product_id', 'week', 'qty']].reset_index(drop=True)

print(f"✅ SALES: {len(sales_long):,} events (censored weeks excluded)")
print(f"   Date range: {sales_long['week'].min()} to {sales_long['week'].max()}")
print(f"   Mean qty: {sales_long['qty'].mean():.2f}")
display(sales_long.head())


✅ SALES: 83,526 events (censored weeks excluded)
   Date range: 2021-04-12 00:00:00 to 2024-04-08 00:00:00
   Mean qty: 3.31


,sales_id,store_id,product_id,week,qty
0,0,0,126,2021-04-12,0.0
1,1,1,124,2021-04-12,13.0
2,2,2,124,2021-04-12,5.0
3,3,3,126,2021-04-12,1.0
4,4,4,124,2021-04-12,10.0


## Build KumoRFM Relational Graph

We'll create a `LocalGraph` connecting Stores, Products, and Sales via foreign keys.


In [9]:
# Create LocalGraph using the handy from_data shortcut
# KumoRFM will auto-infer primary keys, time columns, foreign keys, and semantic types

df_dict = {
    'stores': stores_df,
    'products': products_df,
    'sales': sales_long,
}

graph = rfm.LocalGraph.from_data(df_dict, verbose=True)

print("✅ Built KumoRFM graph")


### 🗂️ Graph Metadata

name,primary_key,time_column
stores,store_id,-
products,product_id,-
sales,sales_id,week


### 🕸️ Graph Links (FK ↔️ PK)

- `sales.product_id` ↔️ `products.product_id`
- `sales.store_id` ↔️ `stores.store_id`

✅ Built KumoRFM graph


In [10]:
# Inspect inferred metadata
print("\n" + "="*80)
print("GRAPH METADATA")
print("="*80)
graph.print_metadata()

print("\n" + "="*80)
print("GRAPH LINKS")
print("="*80)
graph.print_links()



GRAPH METADATA


### 🗂️ Graph Metadata

name,primary_key,time_column
stores,store_id,-
products,product_id,-
sales,sales_id,week



GRAPH LINKS


### 🕸️ Graph Links (FK ↔️ PK)

- `sales.product_id` ↔️ `products.product_id`
- `sales.store_id` ↔️ `stores.store_id`

In [11]:
# Fine-tune semantic types for better model performance
# Ensure qty is numerical (for regression forecasts)
graph['sales']['qty'].stype = 'numerical'
graph['sales']['store_id'].stype = 'ID'
graph['sales']['product_id'].stype = 'ID'

# Product hierarchy should be categorical
graph['products']['Division'].stype = 'categorical'
graph['products']['Department'].stype = 'categorical'
graph['products']['ProductGroup'].stype = 'categorical'

print("✅ Adjusted semantic types")
graph.print_metadata()


✅ Adjusted semantic types


### 🗂️ Graph Metadata

name,primary_key,time_column
stores,store_id,-
products,product_id,-
sales,sales_id,week


In [12]:
# Visualize the graph structure (optional, requires graphviz)
try:
    graph.visualize(show_columns=False)
except Exception as e:
    print(f"⚠️  Visualization requires graphviz: {e}")


⚠️  Visualization requires graphviz: The 'graphviz' package is required for visualization


## Initialize KumoRFM Model

Plug our graph into the KumoRFM foundation model. No training required!


In [13]:
# Initialize KumoRFM model
model = rfm.KumoRFM(graph)
print("✅ KumoRFM model initialized and ready for predictions")


]9;4;3

Output()

]9;4;0✅ KumoRFM model initialized and ready for predictions


## Test: Single SKU Forecast

Test with a single (Store, Product) pair to verify the model works.

**PQL Query**: `PREDICT SUM(sales.qty, 0, 21, days) FOR sales.sales_id IN (...)`

We'll predict the total demand over the next 3 weeks (21 days = protection period).

**Note**: KumoRFM uses `days`, `months`, `years` as time units, not `weeks`.


In [14]:
# Pick a high-volume SKU for testing (Store 64, Product 23 from EDA)
test_store = 64
test_product = 23

# Get the latest sales_id for this SKU
test_sales = sales_long[
    (sales_long['store_id'] == test_store) & 
    (sales_long['product_id'] == test_product)
].sort_values('week', ascending=False)

if len(test_sales) > 0:
    test_sales_id = int(test_sales.iloc[0]['sales_id'])
    test_week = test_sales.iloc[0]['week']
    print(f"Test SKU: Store {test_store}, Product {test_product}")
    print(f"Latest week: {test_week}")
    print(f"sales_id: {test_sales_id}")
else:
    print(f"⚠️  No sales history for Store {test_store}, Product {test_product}")
    test_sales_id = None


Test SKU: Store 64, Product 23
Latest week: 2024-04-08 00:00:00
sales_id: 83516


In [ ]:
# Run a test prediction
if test_sales_id is not None:
    # Note: KumoRFM uses 'days' not 'weeks' - 3 weeks = 21 days
    query_test = f"PREDICT SUM(sales.qty, 0, 21, days) FOR sales.sales_id={test_sales_id}"
    
    print(f"\nPQL Query: {query_test}")
    print("\nRunning prediction (this may take 10-30 seconds)...")
    
    try:
        result_test = model.predict(query_test, run_mode='fast')
        display(result_test)
        
        predicted_demand = result_test['TARGET_PRED'].iloc[0]
        print(f"\n✅ Predicted 3-week (21-day) demand: {predicted_demand:.2f} units")
        
        # Compare with historical 3-week average
        hist_3w_avg = test_sales.head(3)['qty'].sum()
        print(f"Historical 3-week total (most recent): {hist_3w_avg:.0f} units")
        
    except Exception as e:
        print(f"❌ Prediction failed: {e}")
        print("\nPossible issues:")
        print("  - API key not configured")
        print("  - Network/quota limits")
        print("  - Insufficient temporal data")
else:
    print("⚠️  Skipping test prediction (no test SKU found)")



PQL Query: PREDICT SUM(sales.qty, 0, 3, weeks) FOR sales.sales_id=83516

Running prediction (this may take 10-30 seconds)...
❌ Prediction failed: Failed to parse query 'PREDICT SUM(sales.qty, 0, 3, weeks) FOR sales.sales_id=83516'. Unable to process invalid client request: Encountered error while parsing the predictive query: Encountered the following issues during parsing: Errors:
Invalid Syntax. The target (PREDICT) clause in this query is empty or invalid.
Specific error: 
Could not process the text: "weeks"; in line 1 and col 29; Encountered the following error: "no viable alternative at input 'SUM(sales.qty,0,3,weeks'"; 

Possible issues:
  - API key not configured
  - Network/quota limits
  - Insufficient temporal data


## Generate Forecasts for All SKUs

Strategy:
1. Group sales by (store_id, product_id) and get latest sales_id for each SKU
2. Batch predict: `PREDICT SUM(sales.qty, 0, 21, days) FOR sales.sales_id IN (...)` (21 days = 3 weeks)
3. Fallback: use historical average for SKUs without predictions


In [16]:
# Prepare submission SKUs: get latest sales_id for each (store, product)
latest_sales = (
    sales_long.sort_values('week', ascending=False)
    .groupby(['store_id', 'product_id'])
    .first()
    .reset_index()
    [['store_id', 'product_id', 'sales_id', 'week']]
)

# Merge with submission template
submission_skus = template[INDEX].copy()
submission_skus = submission_skus.rename(columns={'Store': 'store_id', 'Product': 'product_id'})
submission_skus = submission_skus.merge(
    latest_sales,
    on=['store_id', 'product_id'],
    how='left'
)

print(f"✅ Prepared {len(submission_skus)} SKUs for forecasting")
print(f"   With sales history: {submission_skus['sales_id'].notna().sum()}")
print(f"   Cold-start (no history): {submission_skus['sales_id'].isna().sum()}")
display(submission_skus.head())


✅ Prepared 599 SKUs for forecasting
   With sales history: 599
   Cold-start (no history): 0


,store_id,product_id,sales_id,week
0,0,126,82927,2024-04-08
1,0,182,82928,2024-04-08
2,1,124,82929,2024-04-08
3,2,124,82930,2024-04-08
4,2,126,82931,2024-04-08


In [17]:
# Batch predictions in chunks (API may have limits)
BATCH_SIZE = 50  # Adjust based on API quota

valid_sales_ids = submission_skus['sales_id'].dropna().astype(int).tolist()
batches = [valid_sales_ids[i:i+BATCH_SIZE] for i in range(0, len(valid_sales_ids), BATCH_SIZE)]

print(f"\n📊 Forecasting plan:")
print(f"   Total SKUs to predict: {len(valid_sales_ids)}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Number of batches: {len(batches)}")
print(f"\n⏱️  Estimated time: {len(batches) * 20 / 60:.1f} minutes (assuming ~20s per batch)")
print("\nNote: This will consume API quota. Consider running with run_mode='fast' first.")



📊 Forecasting plan:
   Total SKUs to predict: 599
   Batch size: 50
   Number of batches: 12

⏱️  Estimated time: 4.0 minutes (assuming ~20s per batch)

Note: This will consume API quota. Consider running with run_mode='fast' first.


In [ ]:
# Run batch predictions
# WARNING: This may take 10-20 minutes depending on API and batch size

all_predictions = []
failed_batches = []

for i, batch in enumerate(batches):
    sales_ids_str = ",".join(map(str, batch))
    # Note: Use 'days' not 'weeks' - 3 weeks = 21 days
    query = f"PREDICT SUM(sales.qty, 0, 21, days) FOR sales.sales_id IN ({sales_ids_str})"
    
    print(f"\nBatch {i+1}/{len(batches)}: {len(batch)} SKUs...", end=" ")
    
    try:
        result = model.predict(query, run_mode='fast')  # Use 'fast' for speed
        all_predictions.append(result)
        print(f"✅ {len(result)} predictions")
    except Exception as e:
        print(f"❌ Failed: {e}")
        failed_batches.append(i)
        continue

# Combine results
if all_predictions:
    forecast_df = pd.concat(all_predictions, ignore_index=True)
    print(f"\n✅ Successfully predicted {len(forecast_df)} SKUs")
    print(f"❌ Failed batches: {len(failed_batches)}")
    display(forecast_df.head())
else:
    print("\n❌ No predictions generated")
    forecast_df = None



Batch 1/12: 50 SKUs... ❌ Failed: Failed to parse query 'PREDICT SUM(sales.qty, 0, 3, weeks) FOR sales.sales_id IN (82927,82928,82929,82930,82931,82932,82933,82934,82935,82936,82937,82938,82939,82940,82941,82942,82943,82944,82945,82946,82947,82948,82949,82950,82951,82952,82953,82954,82955,82956,82957,82958,82959,82960,82961,82962,82963,82964,82965,82966,82967,82968,82969,82970,82971,82972,82973,82974,82975,82976)'. Unable to process invalid client request: Encountered error while parsing the predictive query: Encountered the following issues during parsing: Errors:
Invalid Syntax. The target (PREDICT) clause in this query is empty or invalid.
Specific error: 
Could not process the text: "weeks"; in line 1 and col 29; Encountered the following error: "no viable alternative at input 'SUM(sales.qty,0,3,weeks'"; 

Batch 2/12: 50 SKUs... ❌ Failed: Failed to parse query 'PREDICT SUM(sales.qty, 0, 3, weeks) FOR sales.sales_id IN (82977,82978,82979,82980,82981,82982,82983,82984,82985,82986,829

## Map Forecasts and Apply Fallback

Map KumoRFM forecasts back to SKUs. For cold-start SKUs or failed predictions, use historical 3-week average.


In [19]:
# Extract sales_id -> forecast mapping
if forecast_df is not None and len(forecast_df) > 0:
    # KumoRFM returns ENTITY column with sales_id
    forecast_df['sales_id'] = forecast_df['ENTITY'].astype(int)
    forecast_map = forecast_df.set_index('sales_id')['TARGET_PRED'].to_dict()
    
    submission_skus['kumo_forecast'] = submission_skus['sales_id'].map(forecast_map)
    
    print(f"✅ Mapped KumoRFM forecasts to {submission_skus['kumo_forecast'].notna().sum()} SKUs")
else:
    submission_skus['kumo_forecast'] = np.nan
    print("⚠️  No KumoRFM forecasts available")


⚠️  No KumoRFM forecasts available


In [20]:
# Fallback: historical 3-week average for missing forecasts
hist_avg = (
    sales_c.mean(axis=1, skipna=True).fillna(0.0)
    .reset_index()
    .rename(columns={0: 'weekly_avg', 'Store': 'store_id', 'Product': 'product_id'})
)

submission_skus = submission_skus.merge(hist_avg, on=['store_id', 'product_id'], how='left')
submission_skus['hist_3w_forecast'] = (submission_skus['weekly_avg'] * 3).fillna(0.0)

# Final forecast: KumoRFM if available, else historical fallback
submission_skus['demand_forecast'] = submission_skus['kumo_forecast'].fillna(
    submission_skus['hist_3w_forecast']
).fillna(0.0)

print(f"\n✅ Final forecast summary:")
print(f"   From KumoRFM: {submission_skus['kumo_forecast'].notna().sum()}")
print(f"   From historical fallback: {submission_skus['kumo_forecast'].isna().sum()}")
print(f"   Mean forecast: {submission_skus['demand_forecast'].mean():.2f}")
print(f"   Median forecast: {submission_skus['demand_forecast'].median():.2f}")
print(f"   Max forecast: {submission_skus['demand_forecast'].max():.2f}")



✅ Final forecast summary:
   From KumoRFM: 0
   From historical fallback: 599
   Mean forecast: 9.63
   Median forecast: 3.76
   Max forecast: 296.52


## Convert Forecasts to Orders (Base-Stock Policy)

Order quantity = max(0, forecast - inventory_position)


In [21]:
# Load inventory position
state = initial_state[['Store', 'Product', 'End Inventory', 'In Transit W+1', 'In Transit W+2']].copy()
state = state.rename(columns={
    'Store': 'store_id',
    'Product': 'product_id',
    'End Inventory': 'on_hand',
    'In Transit W+1': 'in_transit_1',
    'In Transit W+2': 'in_transit_2'
})
state['inv_position'] = state['on_hand'] + state['in_transit_1'] + state['in_transit_2']

# Merge with forecasts
submission_skus = submission_skus.merge(
    state[['store_id', 'product_id', 'inv_position']],
    on=['store_id', 'product_id'],
    how='left'
)

print(f"✅ Loaded inventory position")
display(submission_skus[['store_id', 'product_id', 'demand_forecast', 'inv_position']].head())


✅ Loaded inventory position


,store_id,product_id,demand_forecast,inv_position
0,0,126,6.229299,6
1,0,182,2.805556,2
2,1,124,20.800000,12
3,2,124,27.821656,16
4,2,126,7.441558,4


In [22]:
# Compute orders
submission_skus['order_qty'] = (
    submission_skus['demand_forecast'] - submission_skus['inv_position'].fillna(0.0)
).clip(lower=0.0).round().astype(int)

print(f"\n✅ Order summary:")
print(f"   Total units: {submission_skus['order_qty'].sum():,}")
print(f"   Mean: {submission_skus['order_qty'].mean():.2f}")
print(f"   Median: {submission_skus['order_qty'].median():.0f}")
print(f"   Max: {submission_skus['order_qty'].max()}")
print(f"   SKUs with orders > 0: {(submission_skus['order_qty'] > 0).sum()}")

print(f"\n📈 Top 10 orders:")
display(
    submission_skus.nlargest(10, 'order_qty')[
        ['store_id', 'product_id', 'demand_forecast', 'inv_position', 'order_qty']
    ]
)



✅ Order summary:
   Total units: 1,860
   Mean: 3.11
   Median: 1
   Max: 98
   SKUs with orders > 0: 513

📈 Top 10 orders:


,store_id,product_id,demand_forecast,inv_position,order_qty
277,61,124,296.522293,199,98
133,60,125,253.070064,159,94
201,61,23,263.675159,192,72
586,64,17,150.019108,98,52
554,63,124,208.841584,166,43
94,60,23,158.904459,118,41
134,60,126,61.703226,30,32
222,61,48,97.834395,66,32
458,62,126,55.307692,26,29
278,61,126,52.815287,27,26


## Generate Submission CSV


In [23]:
# Prepare final submission in template order
submission_final = template[INDEX].copy()
submission_final = submission_final.merge(
    submission_skus[['store_id', 'product_id', 'order_qty']].rename(
        columns={'store_id': 'Store', 'product_id': 'Product'}
    ),
    on=INDEX,
    how='left'
)

# Fill any missing with 0
submission_final['order_qty'] = submission_final['order_qty'].fillna(0).astype(int)

# Rename to match submission format (column "0")
submission_final = submission_final.rename(columns={'order_qty': '0'})

# Save to CSV
output_path = SUB_DIR / "orders_kumo_rfm.csv"
submission_final.to_csv(output_path, index=False)

print(f"\n" + "="*80)
print("📁 SUBMISSION SAVED")
print("="*80)
print(f"File: {output_path}")
print(f"Rows: {len(submission_final)}")
print(f"Total units: {submission_final['0'].sum():,}")
print(f"\nTop 10 orders:")
display(submission_final.sort_values('0', ascending=False).head(10))



📁 SUBMISSION SAVED
File: /Users/senoni/noni/vn2inventory/submissions/orders_kumo_rfm.csv
Rows: 599
Total units: 1,860

Top 10 orders:


,Store,Product,0
277,61,124,98
133,60,125,94
201,61,23,72
586,64,17,52
554,63,124,43
94,60,23,41
222,61,48,32
134,60,126,32
458,62,126,29
278,61,126,26


## Diagnostic: Compare with Baseline

Compare KumoRFM orders vs. your existing hierarchical Bayes CV solution.


In [24]:
# Compare with hierarchical Bayes baseline (if exists)
try:
    baseline_path = SUB_DIR / "orders_hierarchical_final_store_cv.csv"
    if baseline_path.exists():
        baseline = pd.read_csv(baseline_path)
        baseline = baseline.rename(columns={'0': 'baseline_order'})
        
        comparison = submission_final.merge(
            baseline[INDEX + ['baseline_order']],
            on=INDEX,
            how='left'
        )
        
        comparison['diff'] = comparison['0'] - comparison['baseline_order']
        
        print("="*80)
        print("COMPARISON: KumoRFM vs. Hierarchical Bayes CV")
        print("="*80)
        print(f"  KumoRFM total: {comparison['0'].sum():,} units")
        print(f"  Baseline total: {comparison['baseline_order'].sum():,} units")
        print(f"  Difference: {comparison['diff'].sum():+,} units ({comparison['diff'].sum()/comparison['baseline_order'].sum()*100:+.1f}%)")
        print(f"  Mean absolute diff: {comparison['diff'].abs().mean():.2f}")
        print(f"  Correlation: {comparison[['0', 'baseline_order']].corr().iloc[0,1]:.3f}")
        
        print(f"\nTop 10 SKUs where KumoRFM orders MORE:")
        display(comparison.sort_values('diff', ascending=False).head(10)[INDEX + ['0', 'baseline_order', 'diff']])
        
        print(f"\nTop 10 SKUs where KumoRFM orders LESS:")
        display(comparison.sort_values('diff', ascending=True).head(10)[INDEX + ['0', 'baseline_order', 'diff']])
    else:
        print(f"⚠️  Baseline not found at {baseline_path}")
except Exception as e:
    print(f"⚠️  Comparison failed: {e}")


COMPARISON: KumoRFM vs. Hierarchical Bayes CV
  KumoRFM total: 1,860 units
  Baseline total: 2,123 units
  Difference: -263 units (-12.4%)
  Mean absolute diff: 3.93
  Correlation: 0.258

Top 10 SKUs where KumoRFM orders MORE:


,Store,Product,0,baseline_order,diff
277,61,124,98,0,98
133,60,125,94,0,94
201,61,23,72,0,72
586,64,17,52,0,52
554,63,124,43,0,43
222,61,48,32,0,32
134,60,126,32,0,32
458,62,126,29,0,29
278,61,126,26,0,26
221,61,47,20,0,20



Top 10 SKUs where KumoRFM orders LESS:


,Store,Product,0,baseline_order,diff
589,64,23,24,98,-74
532,63,23,13,48,-35
17,12,124,4,38,-34
585,64,16,5,39,-34
34,25,124,3,37,-34
59,42,17,7,40,-33
67,47,124,5,38,-33
47,34,17,1,34,-33
61,43,17,4,37,-33
132,60,124,3,35,-32


## Summary and Next Steps

### ✅ What We Did

1. Structured VN2 data as a relational graph (Stores, Products, Sales)
2. Applied availability-aware masking (excluded censored OOS weeks)
3. Used KumoRFM foundation model to forecast 3-week demand via PQL
4. Converted forecasts to orders with base-stock policy
5. Generated `submissions/orders_kumo_rfm.csv`

### 🎯 Key Advantages

- **Zero training**: No CV folds, no Optuna, no hyperparameter tuning
- **Multi-table reasoning**: Automatically uses Store/Product hierarchy
- **Temporal foundation model**: Learns demand patterns from graph + timestamps
- **Expressive PQL**: Easy experimentation with filters, windows, aggregations

### 🚀 Potential Improvements

1. **Richer features**: Add seasonality indicators, promotion flags, price
2. **Quantile forecasts**: Run multiple PQL queries for p50, p75, p90 to estimate variance
3. **Ensemble**: Blend KumoRFM with hierarchical Bayes (e.g., weighted average)
4. **Department filters**: Use `WHERE` clauses to condition on product groups
5. **Tune run_mode**: Experiment with `'fast'` / `'normal'` / `'best'` for accuracy/speed tradeoff

### ⚠️ Limitations

- **API quota**: Free tier has rate limits; batch predictions can be slow (~10-20 min for 600 SKUs)
- **Cold-start**: SKUs without history fallback to simple average
- **Black-box**: Less interpretable than hierarchical Bayes GLM

### 📁 Files Generated

- `submissions/orders_kumo_rfm.csv`: Ready for competition submission

### 💡 Next Steps

1. **Run the notebook** with your KumoRFM API key
2. **Compare** with baseline on validation data or simulator
3. **Ensemble**: Try weighted combination (e.g., 50% KumoRFM + 50% Hierarchical Bayes)
4. **Advanced PQL**: Experiment with conditional forecasts, department-specific queries
